In [2]:
import pandas as pd
from pathlib import Path
from time import sleep
from ideam_dhime import download_dhime_data

In [2]:
# Lectura archivo de consulta de estaciones
ruta_estaciones = "../../data/raw/E_precipitacion.csv"
df_estaciones = pd.read_csv(ruta_estaciones)
df_estaciones.head()

,codigo,nombre,categoria,estado,municipio,altitud,longitud,latitud,fecha_instalacion,fecha_suspension
0,35190010,CINTAS LAS [35190010],Pluviográfica,Activa,Sogamoso,3400,-72.867667,5.614139,1971-02-15,NaN
1,23110040,TRIQUE EL [23110040],Pluviométrica,Activa,Puerto Boyacá,179,-74.568342,5.880889,1974-08-15,NaN
2,23125010,MUZO [23125010],Climatológica Ordinaria,Suspendida,Muzo,824,-74.133333,5.533333,1936-01-15,NaN
3,24030100,TERMOPAIPA [24030100],Pluviométrica,Suspendida,Paipa,2580,-73.150000,5.766667,1955-11-15,NaN
4,23120110,BRICENO [23120110],Pluviométrica,Suspendida,Briceño,1500,-73.950000,5.700000,1960-05-15,NaN


In [3]:
# Directorio de salida para datos descargados
download_path = Path("../../data/raw/dhime/precipitacion")
download_path.mkdir(parents=True, exist_ok=True)

In [4]:
# Parámetros generales
departamento = "Boyacá"
parameter = "Precipitación"
variable_code = "Precipitación total mensual"

date_ini = "01/01/1950"
date_fin = "01/05/2026"

resultados = []

In [5]:
for idx, estacion in df_estaciones.iterrows():

    codigo_estacion = str(estacion["codigo"])
    municipio = str(estacion["municipio"])
    nombre = str(estacion["nombre"])

    print(f"\nConsultando {idx} | {codigo_estacion} | {nombre} | {municipio}")

    try:
        csv_final = download_dhime_data(
            download_path=str(download_path),
            station_code=codigo_estacion,
            department=departamento,
            municipality=municipio,
            parameter=parameter,
            variable_code=variable_code,
            date_ini=date_ini,
            date_fin=date_fin,
            time_wait=180
        )

        resultados.append({
            "codigo": codigo_estacion,
            "nombre": nombre,
            "municipio": municipio,
            "estado_descarga": "ok",
            "archivo": str(csv_final),
            "error": None
        })

        print(f"Descarga OK: {csv_final}")

    except Exception as e:

        resultados.append({
            "codigo": codigo_estacion,
            "nombre": nombre,
            "municipio": municipio,
            "estado_descarga": "error",
            "archivo": None,
            "error": str(e)
        })

        print(f"Error en {codigo_estacion}: {e}")

#    sleep(5)

df_resultados = pd.DataFrame(resultados)

df_resultados.to_csv(
    "../../data/raw/dhime/logs/precipitacion_log_descargas.csv",
    index=False,
    encoding="utf-8-sig"
)

df_resultados


Consultando 0 | 35190010 | CINTAS LAS [35190010] | Sogamoso
Descarga OK: ../../data/raw/dhime/precipitacion/35190010-Precipitación_total_mensual-01011950-01052026-final.csv

Consultando 1 | 23110040 | TRIQUE EL [23110040] | Puerto Boyacá
Descarga OK: ../../data/raw/dhime/precipitacion/23110040-Precipitación_total_mensual-01011950-01052026-final.csv

Consultando 2 | 23125010 | MUZO  [23125010] | Muzo
Error en 23125010: No se encontró la estación 23125010 en la lista de Muzo.

Consultando 3 | 24030100 | TERMOPAIPA  [24030100] | Paipa
Descarga OK: ../../data/raw/dhime/precipitacion/24030100-Precipitación_total_mensual-01011950-01052026-final.csv

Consultando 4 | 23120110 | BRICENO  [23120110] | Briceño
Error en 23120110: Fallo definitivo al hacer clic en Seleccionar municipio (Briceño).

Consultando 5 | 24015060 | FABRICA TEXTIL  [24015060] | Samacá
Error en 24015060: No se encontró la estación 24015060 en la lista de Samacá.

Consultando 6 | 24035070 | GUICAN [24035070] | Guicán
Descarg

,codigo,nombre,municipio,estado_descarga,archivo,error
0,35190010,CINTAS LAS [35190010],Sogamoso,ok,../../data/raw/dhime/precipitacion/35190010-Pr...,None
1,23110040,TRIQUE EL [23110040],Puerto Boyacá,ok,../../data/raw/dhime/precipitacion/23110040-Pr...,None
2,23125010,MUZO [23125010],Muzo,error,None,No se encontró la estación 23125010 en la list...
3,24030100,TERMOPAIPA [24030100],Paipa,ok,../../data/raw/dhime/precipitacion/24030100-Pr...,None
4,23120110,BRICENO [23120110],Briceño,error,None,Fallo definitivo al hacer clic en Seleccionar ...
...,...,...,...,...,...,...
206,35070210,PACHAVITA [35070210],Pachavita,ok,../../data/raw/dhime/precipitacion/35070210-Pr...,None
207,23120130,STA BARBARA [23120130],San Pablo De Borbur,ok,../../data/raw/dhime/precipitacion/23120130-Pr...,None
208,24035040,LA COPA [24035040],Toca,ok,../../data/raw/dhime/precipitacion/24035040-Pr...,None
209,35070220,JENESANO [35070220],Jenesano,ok,../../data/raw/dhime/precipitacion/35070220-Pr...,None


In [ ]:
# Elimina archivos redundantes
ruta_datos = Path("../../data/raw/dhime/precipitacion")

eliminados = []

for archivo in ruta_datos.glob("*.csv"):

    nombre = archivo.name

    # conservar log
    if nombre == "precipitacion_log_descargas.csv":
        continue

    # conservar archivos finales
    if nombre.endswith("-final.csv"):
        continue

    archivo.unlink()

    eliminados.append(nombre)

print(f"Archivos eliminados: {len(eliminados)}")

Archivos eliminados: 295
